## 03 — Risk score & SLA breach probability

**Reproducible run:** requires `data/raw/helpdesk_tickets.csv` from `python scripts/generate_synthetic_data.py`. Uses an in-memory SQLite engine as a local stand-in for the warehouse; the production joins live in `SQL/`. The breach model is a transparent statistical score, not a trained classifier — see cell comments. Data is synthetic.

# SLA Risk Scoring for Open Tickets
GitHub Support Operations Analytics - Predictive Risk Model

This notebook calculates real-time risk scores for open tickets using:
- Ticket age (hours since creation)
- Comment activity (engagement level)
- Label weights (security/critical bugs = higher risk)
- Statistical SLA breach probability model

Output: Risk scores stored in Risk_Scores table for real-time dashboard alerts

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Local, reproducible source: build the dims + fact from the
# synthetic CSV in an in-memory SQLite database.
import sqlalchemy
engine = sqlalchemy.create_engine('sqlite://')  # in-memory
raw = pd.read_csv('../data/raw/helpdesk_tickets.csv')
raw['created_at'] = pd.to_datetime(raw['created_at'], format='mixed')
raw['first_response_at'] = pd.to_datetime(raw['first_response_at'], format='mixed')
raw['closed_at'] = pd.to_datetime(raw['closed_at'], format='mixed')
fact = pd.DataFrame({
    'Ticket_Key': raw['Ticket_Key'],
    'Repo_Name': raw['repo_name'],
    'Author_Name': raw['author_name'],
    'Created_At': raw['created_at'],
    'Label': raw['Labels'],
    'Comment_Count': raw['comment_count'],
    'First_Response_At': raw['first_response_at'],
    'Closed_At': raw['closed_at'],
})
fact.to_sql('Fact_Tickets', engine, if_exists='replace', index=False)


2000

In [3]:
# 2. Fetch all OPEN tickets (not yet closed). The star-schema joins
# are collapsed here because the local fact already carries the
# repo/author names; the Postgres/T-SQL joins live in SQL/.
query = """
SELECT Ticket_Key, Repo_Name, Author_Name, Created_At,
       Label, Comment_Count, First_Response_At
FROM Fact_Tickets
WHERE Closed_At IS NULL
"""
df_open = pd.read_sql(query, engine)
# SQLite returns timestamps as text with mixed microsecond precision;
# parse explicitly rather than relying on a single strptime format.
df_open['Created_At'] = pd.to_datetime(df_open['Created_At'], format='mixed')
df_open['First_Response_At'] = pd.to_datetime(df_open['First_Response_At'], format='mixed')
print(f"Loaded {len(df_open)} open tickets")
print(df_open.head())


Loaded 515 open tickets
  Ticket_Key Repo_Name Author_Name          Created_At        Label  \
0    T-00002  repo-014   user-0073 2023-04-27 07:00:00     question   
1    T-00003  repo-032   user-0196 2023-12-26 12:00:00     security   
2    T-00004  repo-003   user-0416 2023-05-28 22:00:00          bug   
3    T-00006  repo-026   user-0458 2023-10-09 07:00:00  enhancement   
4    T-00007  repo-038   user-0307 2023-08-15 17:00:00          bug   

   Comment_Count          First_Response_At  
0              3 2023-04-27 07:46:42.311565  
1              0 2023-12-27 18:46:31.419017  
2              6 2023-05-29 06:13:42.368585  
3              2 2023-10-09 18:35:24.346023  
4             13 2023-08-15 17:30:00.000000  


In [4]:
# 3. Calculate Ticket Age (in hours)
df_open['Ticket_Age_Hours'] = (datetime.now() - df_open['Created_At']).dt.total_seconds() / 3600

print("Ticket Age Statistics:")
print(df_open['Ticket_Age_Hours'].describe())

Ticket Age Statistics:
count      515.000000
mean     28764.507159
std       2531.672707
min      24337.524635
25%      26466.524635
50%      28939.524635
75%      30947.524635
max      33006.524635
Name: Ticket_Age_Hours, dtype: float64


In [5]:
# 4. Assign Label Weights (Security = High Risk)
label_weights = {
    'security': 1.5,
    'critical-bug': 1.4,
    'vulnerability': 1.5,
    'bug': 1.0,
    'enhancement': 0.3,
    'feature': 0.3,
    'question': 0.5,
    'help wanted': 0.6,
    'documentation': 0.2,
    'other': 0.5
}

df_open['Label_Weight'] = df_open['Label'].str.lower().map(label_weights).fillna(0.5)

print("Label Weight Distribution:")
print(df_open.groupby('Label')['Label_Weight'].first().sort_values(ascending=False))

Label Weight Distribution:
Label
vulnerability    1.5
security         1.5
critical-bug     1.4
bug              1.0
help wanted      0.6
question         0.5
other            0.5
enhancement      0.3
feature          0.3
documentation    0.2
Name: Label_Weight, dtype: float64


In [6]:
# 5. Calculate SLA Breach Probability using Statistical Model
# Model: Probability of breaching 24-hour SLA based on:
#   - Ticket age (hours already spent)
#   - Comment count (engagement/complexity indicator)
#   - Label weight (risk multiplier)

def calculate_breach_probability(row):
    """
    Statistical model for SLA breach probability.
    
    Factors:
    - Time spent: How many hours left to SLA deadline (24h total)
    - Comment activity: More comments = higher complexity = higher risk
    - Label weight: Security/critical bugs have inherent higher risk
    """
    
    ticket_age = row['Ticket_Age_Hours']
    comment_count = row['Comment_Count']
    label_weight = row['Label_Weight']
    
    # SLA threshold: 24 hours
    time_remaining = max(0, 24 - ticket_age)
    
    # Base probability: Time-based
    # If 18 hours passed, 6 hours remaining = 75% breach risk (18/24)
    time_risk = ticket_age / 24
    
    # Comment penalty: Each additional comment increases complexity risk
    # Formula: +5% risk per comment (max 40% bonus)
    comment_risk = min(0.40, comment_count * 0.05)
    
    # Label weight multiplier
    # Security bugs have 1.5x weight, so their breach probability increases 50%
    
    # Combined formula (normalized to 0-100)
    breach_prob = ((time_risk * 0.5) + (comment_risk * 0.3) + (label_weight * 0.2)) * 100
    
    return np.clip(breach_prob, 0, 100)

df_open['SLA_Breach_Probability'] = df_open.apply(calculate_breach_probability, axis=1)

print("\nSLA Breach Probability Distribution:")
print(df_open['SLA_Breach_Probability'].describe())


SLA Breach Probability Distribution:
count    515.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: SLA_Breach_Probability, dtype: float64


In [7]:
# 6. Create Risk Score Classification
def classify_risk(breach_prob):
    """Classify tickets into risk tiers"""
    if breach_prob >= 80:
        return 'CRITICAL - Immediate Action'
    elif breach_prob >= 60:
        return 'HIGH - Monitor Closely'
    elif breach_prob >= 40:
        return 'MEDIUM - Track Progress'
    elif breach_prob >= 20:
        return 'LOW - Standard Process'
    else:
        return 'MINIMAL - On Track'

df_open['Risk_Tier'] = df_open['SLA_Breach_Probability'].apply(classify_risk)

print("\nRisk Tier Distribution:")
print(df_open['Risk_Tier'].value_counts())


Risk Tier Distribution:
Risk_Tier
CRITICAL - Immediate Action    515
Name: count, dtype: int64


In [8]:
# 7. Calculate Response Time Factor
# If first response exists, calculate TTFR; if not, use ticket age
df_open['Hours_Since_First_Response'] = df_open.apply(
    lambda row: (datetime.now() - row['First_Response_At']).total_seconds() / 3600 
    if pd.notna(row['First_Response_At']) 
    else np.nan,
    axis=1
)

df_open['Has_Response'] = df_open['First_Response_At'].notna().astype(int)

print(f"\nTickets with First Response: {df_open['Has_Response'].sum()}")
print(f"Tickets Awaiting Response: {(1 - df_open['Has_Response']).sum()}")


Tickets with First Response: 515
Tickets Awaiting Response: 0


In [9]:
# 8. Prepare Risk Score Output
risk_scores_output = df_open[[
    'Ticket_Key',
    'Repo_Name',
    'Author_Name',
    'Created_At',
    'Ticket_Age_Hours',
    'Comment_Count',
    'Label_Weight',
    'Has_Response',
    'SLA_Breach_Probability',
    'Risk_Tier'
]].copy()

risk_scores_output['Calculated_At'] = datetime.now()

print("\nSample Risk Scores (Top 10 Highest Risk):")
print(risk_scores_output.nlargest(10, 'SLA_Breach_Probability')[['Ticket_Key', 'Repo_Name', 'Ticket_Age_Hours', 'Comment_Count', 'SLA_Breach_Probability', 'Risk_Tier']])


Sample Risk Scores (Top 10 Highest Risk):
  Ticket_Key Repo_Name  Ticket_Age_Hours  Comment_Count  \
0    T-00002  repo-014      30251.524635              3   
1    T-00003  repo-032      24414.524635              0   
2    T-00004  repo-003      29492.524635              6   
3    T-00006  repo-026      26291.524635              2   
4    T-00007  repo-038      27601.524635             13   
5    T-00009  repo-036      26650.524635              1   
6    T-00010  repo-037      27679.524635              0   
7    T-00013  repo-038      30905.524635              6   
8    T-00018  repo-022      25590.524635              3   
9    T-00019  repo-029      25146.524635              1   

   SLA_Breach_Probability                    Risk_Tier  
0                   100.0  CRITICAL - Immediate Action  
1                   100.0  CRITICAL - Immediate Action  
2                   100.0  CRITICAL - Immediate Action  
3                   100.0  CRITICAL - Immediate Action  
4                   10

In [10]:
# 9. Upload Risk Scores to SQL
risk_scores_output.to_sql(
    'Risk_Scores',
    con=engine,
    if_exists='replace',
    index=False
)

print("\n✅ Risk Scores uploaded to Risk_Scores table")
print(f"Total records: {len(risk_scores_output)}")
print(f"CRITICAL risk tickets: {(risk_scores_output['Risk_Tier'] == 'CRITICAL - Immediate Action').sum()}")
print(f"HIGH risk tickets: {(risk_scores_output['Risk_Tier'] == 'HIGH - Monitor Closely').sum()}")


✅ Risk Scores uploaded to Risk_Scores table
Total records: 515
CRITICAL risk tickets: 515
HIGH risk tickets: 0


In [11]:
# 10. Summary Statistics
print("\n=== RISK SCORE SUMMARY ===")
print(f"\nTickets at Risk of SLA Breach (>60% prob): {(risk_scores_output['SLA_Breach_Probability'] >= 60).sum()}")
print(f"Average SLA Breach Probability: {risk_scores_output['SLA_Breach_Probability'].mean():.2f}%")
print(f"\nBy Risk Tier:")
print(risk_scores_output['Risk_Tier'].value_counts().to_string())

print(f"\n📊 High-Risk Repositories:")
high_risk_repos = risk_scores_output[risk_scores_output['SLA_Breach_Probability'] >= 60].groupby('Repo_Name').size().sort_values(ascending=False).head()
print(high_risk_repos.to_string())


=== RISK SCORE SUMMARY ===

Tickets at Risk of SLA Breach (>60% prob): 515
Average SLA Breach Probability: 100.00%

By Risk Tier:
Risk_Tier
CRITICAL - Immediate Action    515

📊 High-Risk Repositories:
Repo_Name
repo-012    21
repo-003    20
repo-007    17
repo-001    17
repo-019    17
